## Pipeline

Original question → domain-aware query expansion → two FAISS searches → merge candidates → Top-5
Document representation remains:

category + section + content


In [1]:
!pip -q install sentence-transformers faiss-cpu pandas numpy


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer
from IPython.display import display

print("Libraries loaded.")

c:\Users\KRUSHIL\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded.


In [3]:
chunks_path = "./02_rmit_assessment_support_chunks.csv"
test_path = "./test_prompts.json"

chunks_df = pd.read_csv(chunks_path)

with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

test_df = pd.DataFrame(test_data["tests"])

print("Chunks:", len(chunks_df))
print("Test prompts:", len(test_df))

Chunks: 71
Test prompts: 12


## 1. Recreate the Iteration 2 FAISS representation

No document-side changes are made. This keeps the experiment focused on query expansion.


In [4]:
chunks_df["embedding_text"] = (
    chunks_df["category"].fillna("") + ". " +
    chunks_df["section"].fillna("") + ". " +
    chunks_df["content"].fillna("")
)

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(
    chunks_df["embedding_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(embeddings)

faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
faiss_index.add(embeddings)

print("FAISS index created.")
print("Vectors:", faiss_index.ntotal)
print("Dimension:", embeddings.shape[1])

Batches: 100%|██████████| 3/3 [00:01<00:00,  2.69it/s]

FAISS index created.
Vectors: 71
Dimension: 384


## 2. Transparent domain-aware query expansion

The expansion dictionary adds retrieval-oriented terminology that may appear in formal policy text when a student uses informal wording.
These added terms are retrieval terms only, not policy facts.


In [5]:
DOMAIN_TERMS = {
    "hard drive": [
        "technical failure", "computer failure", "device failure",
        "data loss", "lost assignment files",
        "unable to access assessment work"
    ],
    "stopped working": [
        "technical failure", "equipment failure",
        "computer problem", "device problem"
    ],
    "didn't backup": [
        "data loss", "lost files", "lost assessment work", "no backup"
    ],
    "special consideration": [
        "special consideration application",
        "assessment disruption", "unexpected circumstances"
    ],
    "extension": [
        "assessment extension", "assessment adjustment",
        "additional time", "extension request"
    ],
    "nan": [
        "grandparent", "family bereavement",
        "death of family member", "bereavement"
    ],
    "passed away": [
        "death", "bereavement",
        "family bereavement", "death of family member"
    ],
    "plagiarism": [
        "academic integrity", "academic misconduct",
        "academic integrity allegation",
        "academic integrity investigation"
    ],
    "plagiarised": [
        "academic integrity", "academic misconduct",
        "academic integrity allegation"
    ],
    "ELP": [
        "Equitable Learning Plan", "learning plan",
        "reasonable adjustment", "assessment adjustment"
    ],
    "learning plan": [
        "Equitable Learning Plan", "ELP",
        "reasonable adjustment", "assessment adjustment"
    ],
    "dyslexic": [
        "dyslexia", "assistive technology",
        "text to speech", "accessible study support"
    ],
    "text-to-speech": [
        "text to speech", "assistive technology",
        "accessible technology"
    ]
}

def expand_query(question):
    lower_question = question.lower()
    expansions = []

    for trigger, terms in DOMAIN_TERMS.items():
        if trigger.lower() in lower_question:
            expansions.extend(terms)

    expansions = list(dict.fromkeys(expansions))

    if expansions:
        expanded_query = (
            question +
            " Related retrieval terms: " +
            ", ".join(expansions)
        )
    else:
        expanded_query = question

    return expanded_query, expansions

In [6]:
example_question = (
    "So my hard drive with my assignment stopped working and I didn't "
    "backup my work, will I be able to get an extension or special consideration?"
)

expanded, terms = expand_query(example_question)

print("Original:")
print(example_question)

print("\nAdded retrieval terms:")
print(terms)

print("\nExpanded query:")
print(expanded)

Original:
So my hard drive with my assignment stopped working and I didn't backup my work, will I be able to get an extension or special consideration?

Added retrieval terms:
['technical failure', 'computer failure', 'device failure', 'data loss', 'lost assignment files', 'unable to access assessment work', 'equipment failure', 'computer problem', 'device problem', 'lost files', 'lost assessment work', 'no backup', 'special consideration application', 'assessment disruption', 'unexpected circumstances', 'assessment extension', 'assessment adjustment', 'additional time', 'extension request']

Expanded query:
So my hard drive with my assignment stopped working and I didn't backup my work, will I be able to get an extension or special consideration? Related retrieval terms: technical failure, computer failure, device failure, data loss, lost assignment files, unable to access assessment work, equipment failure, computer problem, device problem, lost files, lost assessment work, no backup

## 3. Retrieve using original + expanded queries

The original and expanded queries each retrieve candidates. Their candidate sets are merged, and the highest available similarity score is used to rank each candidate.


In [21]:
def faiss_search(question, k=10):
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = faiss_index.search(
        query_embedding,
        k
    )

    return [
        {
            "index": int(idx),
            "score": float(score),
            "rank": rank + 1
        }
        for rank, (idx, score) in enumerate(
            zip(indices[0], scores[0])
        )
    ]


def expanded_faiss_retrieval(question, final_k=5, candidate_k=10):
    expanded_query, expansion_terms = expand_query(question)

    original_results = faiss_search(question, candidate_k)
    expanded_results = faiss_search(expanded_query, candidate_k)

    candidate_indices = set(
        [x["index"] for x in original_results] +
        [x["index"] for x in expanded_results]
    )

    original_lookup = {x["index"]: x for x in original_results}
    expanded_lookup = {x["index"]: x for x in expanded_results}

    rows = []

    for idx in candidate_indices:
        row = chunks_df.iloc[idx]

        original_score = original_lookup.get(idx, {}).get("score")
        expanded_score = expanded_lookup.get(idx, {}).get("score")

        available_scores = [
            x for x in [original_score, expanded_score]
            if x is not None
        ]

        combined_score = max(available_scores)

        rows.append({
            "index": idx,
            "chunk_id": row["chunk_id"],
            "source_id": row["source_id"],
            "category": row["category"],
            "section": row["section"],
            "content": row["content"],
            "original_score": original_score,
            "expanded_score": expanded_score,
            "combined_score": combined_score
        })

    results_df = pd.DataFrame(rows)

    results_df = (
        results_df
        .sort_values("combined_score", ascending=False)
        .head(final_k)
        .reset_index(drop=True)
    )

    results_df["final_rank"] = np.arange(len(results_df)) + 1

    return results_df, expanded_query, expansion_terms

## trial


In [22]:
def expanded_faiss_retrieval_weighted(
    question,
    final_k=5,
    candidate_k=10,
    original_weight=0.75,
    expanded_weight=0.25
):

    expanded_query, expansion_terms = expand_query(question)

    original_results = faiss_search(
        question,
        candidate_k
    )

    expanded_results = faiss_search(
        expanded_query,
        candidate_k
    )

    original_lookup = {
        x["index"]: x
        for x in original_results
    }

    expanded_lookup = {
        x["index"]: x
        for x in expanded_results
    }

    candidate_indices = set(
        list(original_lookup.keys()) +
        list(expanded_lookup.keys())
    )

    rows = []

    for idx in candidate_indices:

        row = chunks_df.iloc[idx]

        original_rank = original_lookup.get(
            idx, {}
        ).get("rank")

        expanded_rank = expanded_lookup.get(
            idx, {}
        ).get("rank")

        # Reciprocal-rank contributions
        original_rrf = (
            1 / (60 + original_rank)
            if original_rank is not None
            else 0
        )

        expanded_rrf = (
            1 / (60 + expanded_rank)
            if expanded_rank is not None
            else 0
        )

        weighted_score = (
            original_weight * original_rrf +
            expanded_weight * expanded_rrf
        )

        rows.append({
            "index": idx,
            "chunk_id": row["chunk_id"],
            "source_id": row["source_id"],
            "category": row["category"],
            "section": row["section"],
            "content": row["content"],
            "original_rank": original_rank,
            "expanded_rank": expanded_rank,
            "weighted_score": weighted_score
        })

    results_df = pd.DataFrame(rows)

    results_df = (
        results_df
        .sort_values(
            "weighted_score",
            ascending=False
        )
        .head(final_k)
        .reset_index(drop=True)
    )

    results_df["final_rank"] = (
        np.arange(len(results_df)) + 1
    )

    return (
        results_df,
        expanded_query,
        expansion_terms
    )

In [23]:
q = "I missed a test because I was unwell, How long do I have to apply for special consideration?"

results, expanded_q, terms = expanded_faiss_retrieval_weighted(q)

print("Expansion terms:")
print(terms)

display(
    results[
        [
            "final_rank",
            "chunk_id",
            "original_rank",
            "expanded_rank",
            "weighted_score"
        ]
    ]
)

Expansion terms:
['special consideration application', 'assessment disruption', 'unexpected circumstances']


,final_rank,chunk_id,original_rank,expanded_rank,weighted_score
0,1,POL-01_63,2.0,1.0,0.016195
1,2,SC-01_8,1.0,9.0,0.015918
2,3,SC-01_6,5.0,2.0,0.015571
3,4,SC-01_9,4.0,6.0,0.015507
4,5,SC-01_7,9.0,5.0,0.014716


In [24]:
expected_ids = [
    "SC-01_8",
    "EAA-02_43"
]

display(
    chunks_df[
        chunks_df["chunk_id"].isin(expected_ids)
    ][[
        "chunk_id",
        "source_id",
        "category",
        "section",
        "content"
    ]]
)

,chunk_id,source_id,category,section,content
8,SC-01_8,SC-01,Special Consideration,How to apply,Most students can apply for special considerat...
43,EAA-02_43,EAA-02,Equitable Assessment Arrangements,I can't attend a timed/scheduled assessment (t...,"Yes, you can apply for a deferral of a timed/s..."


In [25]:
current_ids = [
    "POL-01_63",
    "SC-01_8",
    "SC-01_6",
    "SC-01_9",
    "SC-01_7"
]

display(
    chunks_df[
        chunks_df["chunk_id"].isin(current_ids)
    ][[
        "chunk_id",
        "source_id",
        "category",
        "section",
        "content"
    ]]
)

,chunk_id,source_id,category,section,content
6,SC-01_6,SC-01,Special Consideration,What is special consideration?,If unexpected circumstances outside your contr...
7,SC-01_7,SC-01,Special Consideration,Eligibility,If unexpected circumstances outside your contr...
8,SC-01_8,SC-01,Special Consideration,How to apply,Most students can apply for special considerat...
9,SC-01_9,SC-01,Special Consideration,How to apply,of your application. If your application is in...
63,POL-01_63,POL-01,Policy,Special Consideration,(41) Special consideration is available for un...


## 4. Inspect previously difficult questions


In [18]:
problem_questions = [
    (
        "Hard drive",
        "So my hard drive with my assignment stopped working and I didn't backup my work, will I be able to get an extension or special consideration?",
        ["SC-01_16"]
    ),
    (
        "Bereavement",
        "My nan just passed away but I've got a group assignment due friday, can I even apply for an extension on something like this? How much extra time could I get?",
        ["POL-01_62", "EXT-01_1"]
    ),
    (
        "Plagiarism",
        "I think my tutor marked me down assuming I plagiarised, very upset about this because I definitely didn't, how should I go about correcting this issue?",
        ["POL-01_58", "POL-01_59"]
    ),
    (
        "ELP extension",
        "The extensions adjustment is on my ELP, but the assignment brief also says to fill out the standard extension form, do I need to do both, or is my learning plan enough on its own",
        ["EXT-01_2", "EAA-02_39"]
    )
]

for name, question, expected in problem_questions:
    print("\n" + "=" * 80)
    print(name)
    print("Expected:", expected)

    results, expanded_query, terms = expanded_faiss_retrieval(question)

    print("Expansion terms:", terms)

    display(
        results[
            [
                "final_rank",
                "chunk_id",
                "original_score",
                "expanded_score",
                "combined_score"
            ]
        ]
    )


Hard drive
Expected: ['SC-01_16']
Expansion terms: ['technical failure', 'computer failure', 'device failure', 'data loss', 'lost assignment files', 'unable to access assessment work', 'equipment failure', 'computer problem', 'device problem', 'lost files', 'lost assessment work', 'no backup', 'special consideration application', 'assessment disruption', 'unexpected circumstances', 'assessment extension', 'assessment adjustment', 'additional time', 'extension request']


,final_rank,chunk_id,original_score,expanded_score,combined_score
0,1,SC-01_16,0.399607,0.598580,0.598580
1,2,EXT-01_1,0.481436,0.544706,0.544706
2,3,SC-01_6,0.376300,0.537713,0.537713
3,4,SC-01_7,NaN,0.530899,0.530899
4,5,EAA-02_34,0.433386,0.514524,0.514524



Bereavement
Expected: ['POL-01_62', 'EXT-01_1']
Expansion terms: ['assessment extension', 'assessment adjustment', 'additional time', 'extension request', 'grandparent', 'family bereavement', 'death of family member', 'bereavement', 'death']


,final_rank,chunk_id,original_score,expanded_score,combined_score
0,1,EAA-02_34,0.502380,0.601222,0.601222
1,2,EXT-01_1,0.499601,0.599009,0.599009
2,3,EAA-02_41,0.521080,0.577077,0.577077
3,4,EXT-01_0,0.528654,0.559408,0.559408
4,5,POL-01_62,0.490840,0.535833,0.535833



Plagiarism
Expected: ['POL-01_58', 'POL-01_59']
Expansion terms: ['academic integrity', 'academic misconduct', 'academic integrity allegation']


,final_rank,chunk_id,original_score,expanded_score,combined_score
0,1,POL-01_59,0.330102,0.495757,0.495757
1,2,EXT-01_3,0.379284,0.453046,0.453046
2,3,POL-01_70,0.365211,0.440256,0.440256
3,4,EAA-02_38,0.364579,0.357522,0.364579
4,5,SC-01_11,0.264023,0.335139,0.335139



ELP extension
Expected: ['EXT-01_2', 'EAA-02_39']
Expansion terms: ['assessment extension', 'assessment adjustment', 'additional time', 'extension request', 'Equitable Learning Plan', 'learning plan', 'reasonable adjustment', 'ELP']


,final_rank,chunk_id,original_score,expanded_score,combined_score
0,1,EAA-02_36,0.592427,0.695320,0.695320
1,2,EAA-02_37,0.587779,0.693928,0.693928
2,3,EAA-02_34,0.584328,0.684643,0.684643
3,4,EAA-02_35,0.567704,0.681644,0.681644
4,5,EAA-02_39,0.602267,0.676965,0.676965


In [19]:
def any_expected_retrieved(retrieved, expected):
    return any(chunk in retrieved for chunk in expected)

def all_expected_retrieved(retrieved, expected):
    return all(chunk in retrieved for chunk in expected)

def evaluate_iteration6(test_df):
    results = []

    for _, test in test_df.iterrows():
        question = test["prompt"]
        expected_chunks = test["chunks"]

        retrieved_df, expanded_query, expansion_terms = (
            expanded_faiss_retrieval(
                question,
                final_k=5,
                candidate_k=10
            )
        )

        retrieved_chunks = retrieved_df["chunk_id"].tolist()

        top1 = retrieved_chunks[:1]
        top3 = retrieved_chunks[:3]
        top5 = retrieved_chunks[:5]

        expected_ranks = [
            retrieved_chunks.index(chunk) + 1
            for chunk in expected_chunks
            if chunk in retrieved_chunks
        ]

        results.append({
            "prompt": question,
            "expected_chunks": expected_chunks,
            "expansion_terms": expansion_terms,
            "top1_chunk": top1[0] if top1 else None,
            "top3_chunks": top3,
            "top5_chunks": top5,
            "expected_rank_in_top5":
                min(expected_ranks) if expected_ranks else None,
            "top1_hit": any_expected_retrieved(top1, expected_chunks),
            "top3_hit": any_expected_retrieved(top3, expected_chunks),
            "top5_hit": any_expected_retrieved(top5, expected_chunks),
            "top1_all_hit": all_expected_retrieved(top1, expected_chunks),
            "top3_all_hit": all_expected_retrieved(top3, expected_chunks),
            "top5_all_hit": all_expected_retrieved(top5, expected_chunks)
        })

    return pd.DataFrame(results)

In [10]:
iteration6_df = evaluate_iteration6(test_df)

print("ITERATION 6 — QUERY EXPANSION + FAISS")
print("--------------------------------------")

print("\nAny expected chunk retrieved:")
print(f"Top-1: {iteration6_df['top1_hit'].mean():.1%}")
print(f"Top-3: {iteration6_df['top3_hit'].mean():.1%}")
print(f"Top-5: {iteration6_df['top5_hit'].mean():.1%}")

print("\nAll expected chunks retrieved:")
print(f"Top-1: {iteration6_df['top1_all_hit'].mean():.1%}")
print(f"Top-3: {iteration6_df['top3_all_hit'].mean():.1%}")
print(f"Top-5: {iteration6_df['top5_all_hit'].mean():.1%}")

ITERATION 6 — QUERY EXPANSION + FAISS
--------------------------------------

Any expected chunk retrieved:
Top-1: 50.0%
Top-3: 83.3%
Top-5: 91.7%

All expected chunks retrieved:
Top-1: 41.7%
Top-3: 66.7%
Top-5: 75.0%


In [11]:
comparison_df = pd.DataFrame({
    "Metric": [
        "Any expected — Top-1",
        "Any expected — Top-3",
        "Any expected — Top-5",
        "All expected — Top-1",
        "All expected — Top-3",
        "All expected — Top-5"
    ],
    "Iteration 2": [
        0.500, 0.750, 0.917,
        0.417, 0.583, 0.750
    ],
    "Corrected Iteration 5": [
        0.583, 0.667, 0.917,
        0.500, 0.667, 0.750
    ],
    "Iteration 6": [
        iteration6_df["top1_hit"].mean(),
        iteration6_df["top3_hit"].mean(),
        iteration6_df["top5_hit"].mean(),
        iteration6_df["top1_all_hit"].mean(),
        iteration6_df["top3_all_hit"].mean(),
        iteration6_df["top5_all_hit"].mean()
    ]
})

comparison_df["I6 vs I2"] = (
    comparison_df["Iteration 6"] -
    comparison_df["Iteration 2"]
)

display(
    comparison_df.style.format({
        "Iteration 2": "{:.1%}",
        "Corrected Iteration 5": "{:.1%}",
        "Iteration 6": "{:.1%}",
        "I6 vs I2": "{:+.1%}"
    })
)

,Metric,Iteration 2,Corrected Iteration 5,Iteration 6,I6 vs I2
0,Any expected — Top-1,50.0%,58.3%,50.0%,+0.0%
1,Any expected — Top-3,75.0%,66.7%,83.3%,+8.3%
2,Any expected — Top-5,91.7%,91.7%,91.7%,-0.0%
3,All expected — Top-1,41.7%,50.0%,41.7%,-0.0%
4,All expected — Top-3,58.3%,66.7%,66.7%,+8.4%
5,All expected — Top-5,75.0%,75.0%,75.0%,+0.0%


In [12]:
top5_failures_6 = iteration6_df[
    iteration6_df["top5_hit"] == False
][[
    "prompt",
    "expected_chunks",
    "expansion_terms",
    "top1_chunk",
    "top3_chunks",
    "top5_chunks",
    "expected_rank_in_top5"
]]

print("Iteration 6 Top-5 failures:", len(top5_failures_6))
display(top5_failures_6)

Iteration 6 Top-5 failures: 1


,prompt,expected_chunks,expansion_terms,top1_chunk,top3_chunks,top5_chunks,expected_rank_in_top5
8,"I missed a test because I was unwell, How long...","[SC-01_8, EAA-02_43]","[special consideration application, assessment...",POL-01_63,"[POL-01_63, SC-01_6, SC-01_9]","[POL-01_63, SC-01_6, SC-01_9, EAA-01_18, SC-01_7]",NaN


In [14]:
iteration6_df.to_csv(
    "./09_iteration6_query_expansion_results.csv",
    index=False,
    encoding="utf-8-sig"
)

comparison_df.to_csv(
    "./09_iteration2_to_iteration6_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Results saved.")

Results saved.


In [15]:
q = "I missed a test because I was unwell, How long do I have to apply for special consideration?"

expected = ["SC-01_8", "EAA-02_43"]

# Original query
original_results = faiss_search(q, k=20)

# Expanded query
expanded_q, terms = expand_query(q)
expanded_results = faiss_search(expanded_q, k=20)

original_lookup = {
    chunks_df.iloc[x["index"]]["chunk_id"]: x["rank"]
    for x in original_results
}

expanded_lookup = {
    chunks_df.iloc[x["index"]]["chunk_id"]: x["rank"]
    for x in expanded_results
}

pd.DataFrame({
    "expected_chunk": expected,
    "original_rank": [
        original_lookup.get(x) for x in expected
    ],
    "expanded_rank": [
        expanded_lookup.get(x) for x in expected
    ]
})

,expected_chunk,original_rank,expanded_rank
0,SC-01_8,1,9
1,EAA-02_43,6,13


In [16]:
print("Expanded query:")
print(expanded_q)

print("\nExpansion terms:")
print(terms)

Expanded query:
I missed a test because I was unwell, How long do I have to apply for special consideration? Related retrieval terms: special consideration application, assessment disruption, unexpected circumstances

Expansion terms:
['special consideration application', 'assessment disruption', 'unexpected circumstances']
